In [13]:
import pandas as pd
import numpy as np
import sqlite3
import os

# TASK 1 : DATA INGESTION


In [14]:
def load_data():
    print("TASK 1: DATA INGESTION")

    sales = pd.read_csv("sales_data.csv")
    products = pd.read_csv("products.csv")
    stores = pd.read_csv("stores.csv")

    # shape and first 5 rows 
    for df in [sales,products,stores]:
        print(f"Shape: {df.shape}")
        print(df.head())
        print("\n")

    # Check missing values
    for name, df in [("sales_data", sales ), ("products", products ), ("stores", stores )]:
        print(f"\n{name}:\n{df.isnull().sum()}")

    return sales, products,stores

# TASK 2 : DATA CLEANING

In [15]:
def clean_data(sales ):
    print("\n")
    print("TASK 2: DATA CLEANING")

    # Remove duplicate 
    before = len(sales )
    sales = sales .drop_duplicates()
    after = len(sales )
    print(f"\n Duplicates removed : {before - after} rows")

    # Fill missing quantity with 0,drop rows where amount is null
    sales["quantity"] = sales["quantity"].fillna(0)
    sales = sales.dropna(subset=["amount"])
    print(f"Cleaned DataFrame shape : {sales.shape}")

    # Convert sale_date → datetime, amount → float
    sales["sale_date"] = pd.to_datetime(sales["sale_date"])
    sales["amount"] = sales["amount"].astype(float)

    return sales 

# TASK 3 : DATA TRANSFORMATION

In [16]:
def transform_data(sales , stores , products ):
    print("\n")
    print("TASK 3: DATA TRANSFORMATION")

    # Merge DataFrames
    merged= (sales.merge(stores, on="store_id", how="left").merge(products, on="product_id", how="left"))
    print(merged.to_string(index=False))

    # Add total_revenue column
    merged["total_revenue"] = merged["quantity"] * merged["price"]
    tr = merged["total_revenue"]
    print(f"\ntotal_revenue stats:")
    print(f"Mean: {np.mean(tr):.2f}")
    print(f"Max: {np.max(tr):.2f}")
    print(f"Min: {np.min(tr):.2f}")

    # Revenue by city
    city_revenue = (merged .groupby("city")["total_revenue"].sum().reset_index().sort_values("total_revenue", ascending=False))
    print(f"\nTotal Revenue by City:\n{city_revenue.to_string(index=False)}")

    return merged, city_revenue

# TASK 4 : DATA LOADING (SQL)

In [17]:
def load_to_db(merged ):
    print("\n")
    print("TASK 4: DATA LOADING TO SQLite")

    # Write to SQLite
    conn = sqlite3.connect("retailmart.db")
    merged .to_sql("retail_sales", conn, if_exists="replace", index=False)
    print("\nData loaded into 'retail_sales' table in retailmart.db ")

    # Top 3 best-selling products
    query_top3 = """SELECT product_name, SUM(quantity) AS total_qty_sold FROM retail_sales GROUP  BY product_name ORDER  BY total_qty_sold DESC LIMIT 3"""
    top3 = pd.read_sql_query(query_top3, conn)
    print(f"\nTop 3 Best-Selling Products:\n{top3}")

    return conn, top3

# TASK 5 : REPORTING & INSIGHTS

In [18]:
def reporting(conn, merged, city_revenue, top3):
    print("\n")
    print("TASK 5: REPORTING & INSIGHTS")

    # Revenue per store per day 
    query_store_day = """
    SELECT store_name,
        DATE(sale_date) AS sale_date,
        SUM(total_revenue) AS daily_revenue
        FROM   retail_sales
        GROUP  BY store_name, DATE(sale_date)
        
    """
    store_day = pd.read_sql_query(query_store_day, conn)
    print(f"\nRevenue per Store per Day:\n{store_day}")

    # Summary Report (Python)
    total_txns = len(merged )
    total_rev = merged ["total_revenue"].sum()
    top_city = city_revenue.iloc[0]["city"]
    top_product = top3.iloc[0]["product_name"]

    print("\n Summary report ")

    print(f"Total Transactions : {total_txns}")
    print(f"Total Revenue      : ₹{total_rev:.2f}")
    print(f"Top Selling City   : {top_city}")
    print(f"Top Selling Product: {top_product}")


# TASK 6 : PIPELINE + ERROR HANDLING

In [19]:
def run_pipeline():
    
    print("\n")
    print("TASK 6: RUNNING FULL PIPELINE run_pipeline()")
    conn = None  
    try:
        required_files = ["sales_data.csv", "products.csv", "stores.csv"]
        for file in required_files:
            if not os.path.exists(file):
                raise FileNotFoundError(
                    f"Required file '{file}' not found! "
                    f"Please place it in the current directory."
                )
        # Step 2: 
        sales, products, stores = load_data()
        # Step 3:
        sales = clean_data(sales )
        # Step 4:
        merged, city_revenue = transform_data(sales, stores, products)
        # Step 5:
        conn, top3 = load_to_db(merged )
        # Step 6: 
        reporting(conn, merged, city_revenue, top3)
        print("\nPipeline completed successfully!")

    except FileNotFoundError as e:
        print(f"\nFile Error : {e}")
        print(" Please ensure all CSV files are present and try again.")

    finally:
        if conn:
            conn.close()
            print(" Database connection closed.")


if __name__ == "__main__":
    run_pipeline()




TASK 6: RUNNING FULL PIPELINE run_pipeline()
TASK 1: DATA INGESTION
Shape: (47, 6)
   sale_id store_id product_id  quantity   sale_date  amount
0        1      S01        P01       3.0  2024-01-10  450.00
1        2      S01        P02       1.0  2024-01-10  199.99
2        3      S02        P03       2.0  2024-01-10     NaN
3        4      S02        P01       5.0  2024-01-11  750.00
4        5      S03        P04       NaN  2024-01-11  300.00


Shape: (5, 4)
  product_id          product_name       category   price
0        P01      Basmati Rice 5kg        Grocery  150.00
1        P02  Surf Excel Detergent      Home Care  199.99
2        P03      Amul Butter 500g          Dairy  180.00
3        P04    Colgate Toothpaste  Personal Care   50.00
4        P05         Bournvita 1kg      Beverages  150.00


Shape: (5, 4)
  store_id              store_name       city region
0      S01      RetailMart Andheri     Mumbai   West
1      S02    RetailMart Connaught      Delhi  North
2      S03

C:\Users\Jignesh\AppData\Local\Temp\ipykernel_11404\1200688019.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales["quantity"] = sales["quantity"].fillna(0)
